In [ ]:
import os
import os.path as op

import openneuro
from mne.datasets import sample

from mne_bids import (
    BIDSPath,
    find_matching_paths,
    get_entity_vals,
    make_report,
    print_dir_tree,
    read_raw_bids,
)
import os
from os import path
import openneuro
from mne.datasets import sample
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.viz import plot_topomap

from naplib.io import load_bids

In [ ]:
import mne  # Import MNE

# Download the sample dataset if it is not already downloaded
data_path = mne.datasets.sample.data_path(download=True)

print("MNE sample dataset path:", data_path)

In [ ]:
import os
print(os.path.exists('/Users/uarc/Downloads/ds002778/sub-pd5/ses-off/eeg'))

In [ ]:
# Define the dataset ID
dataset = "ds002778"

# Define the root directory for storing the BIDS dataset
bids_root = op.join(op.dirname(sample.data_path()), dataset)

# Create the directory if it doesn't exist
if not op.isdir(bids_root):
    os.makedirs(bids_root)

In [ ]:
# Download the entire dataset (all subjects, both PD and HC)
openneuro.download(dataset=dataset, target_dir=bids_root)

In [ ]:
from mne_bids import print_dir_tree

print_dir_tree(bids_root, max_depth=4)


In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "pd5"
session = "off"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

This warning message from MNE-BIDS indicates that certain metadata fields in the BIDS dataset (like gender, MMSE, NAART, etc.) cannot be automatically mapped to MNE's data structure.

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
print(raw.info)


raw.notch_filter(freqs=[50, 60])


In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded


Apply Bandpass Filter (Keep Only Relevant Brainwaves)

In [ ]:
# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)


In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions




In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts


In [ ]:
# Extract EEG Features
# Since you are analyzing Alpha/Theta ratio, extract frequency bands:
# Computes Alpha/Theta Ratio per electrode.

from mne import compute_psd_welch
import numpy as np

# Compute Power Spectral Density (PSD)
psds = compute_psd_welch(raw, fmin=1, fmax=40, n_fft=1024).get_data()

# Define Alpha (8-12 Hz) and Theta (4-7 Hz) bands
alpha_idx = np.logical_and(freqs >= 8, freqs <= 12)
theta_idx = np.logical_and(freqs >= 4, freqs <= 7)

# Compute Alpha/Theta Ratio
alpha_power = psds[:, alpha_idx].mean(axis=1)
theta_power = psds[:, theta_idx].mean(axis=1)
alpha_theta_ratio = np.log(alpha_power / theta_power)  # Log transform

print("Alpha/Theta Ratio per Channel:", alpha_theta_ratio)




In [ ]:
# Visualize the Alpha/Theta Ratio as a Topomap
# Generates a topographic map of Alpha/Theta ratio.

import matplotlib.pyplot as plt
from mne.viz import plot_topomap

fig, ax = plt.subplots()
ax.set_title('Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio, raw.info, axes=ax)
plt.show()




## Feature Extraction for Machine Learning

To classify Parkinson’s vs. Healthy, extract features:

    Power Spectral Density (PSD)
    Alpha/Theta Ratio
    Functional Connectivity
    Wavelet Features
    Hjorth Parameters

In [ ]:
# Convert features to a DataFrame for ML models:

import pandas as pd

df = pd.DataFrame({
    'subject': subject,
    'alpha_theta_ratio': alpha_theta_ratio
})

df.to_csv('features.csv', index=False)




## Train a Machine Learning Model

In [ ]:
# Use Scikit-Learn to classify PD vs. HC:
# Trains a random forest classifier on extracted EEG features.

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Load features
data = pd.read_csv('features.csv')

# Split data
X_train, X_test, y_train, y_test = train_test_split(data.drop(columns=['subject']), labels, test_size=0.2, random_state=42)

# Train model
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

# Evaluate
accuracy = clf.score(X_test, y_test)
print(f"Accuracy: {accuracy:.2f}")


